In [1]:
import pandas as pd

movies = pd.read_csv(
    "data/movies.csv",
    # sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1",
    skiprows=1,
    header=None
)

print(movies.head())

   movie_id                               title  \
0         1                    Toy Story (1995)   
1         2                      Jumanji (1995)   
2         3             Grumpier Old Men (1995)   
3         4            Waiting to Exhale (1995)   
4         5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [2]:
movies["genres"] = movies["genres"].apply(lambda x: x.split("|"))

In [3]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)")
movies["title"] = movies["title"].str.replace(r"\(\d{4}\)", "", regex=True).str.strip()

In [4]:
movies

,movie_id,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017
9739,193585,Flint,[Drama],2017
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018


Ratings.dat 읽기

In [5]:
ratings = pd.read_csv(
    "data/ratings.csv",
    # sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    skiprows=1,
    header=None
)

ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

In [6]:
ratings

,user_id,movie_id,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [7]:
# avg_ratings = ratings.groupby("movie_id")["rating"]..mean().reset_index()
avg_ratings = (
    ratings.groupby("movie_id")
    .agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    )
    .reset_index()
)
avg_ratings.rename(columns={"rating": "avg_rating"}, inplace=True)

In [8]:
avg_ratings

,movie_id,avg_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49
...,...,...,...
9719,193581,4.000000,1
9720,193583,3.500000,1
9721,193585,3.500000,1
9722,193587,3.500000,1


In [9]:
movies = movies.merge(avg_ratings, on="movie_id", how="left")

In [10]:
movies

,movie_id,title,genres,year,avg_rating,rating_count
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,3.920930,215.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,3.431818,110.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,3.259615,52.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,2.357143,7.0
4,5,Father of the Bride Part II,[Comedy],1995,3.071429,49.0
...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017,4.000000,1.0
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017,3.500000,1.0
9739,193585,Flint,[Drama],2017,3.500000,1.0
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018,3.500000,1.0


In [11]:
tags = pd.read_csv(
    "data/tags.csv",
    # sep="::",
    engine="python",
    names=["user_id", "movie_id", "tag", "timestamp"],
    skiprows=1,
    header=None
)

tags["tag"] =tags["tag"].str.lower().str.strip()
tags = tags[tags["tag"] != ""]
tags = tags.dropna(subset=["movie_id", "tag"])
tags["tag"].apply(type).value_counts()

tag
<class 'str'>    3683
Name: count, dtype: int64

In [12]:
tags_agg = (tags.groupby("movie_id")["tag"].apply(lambda x: " | ".join(sorted(set(x)))).reset_index().rename(columns={"tag": "tags"}))

tags_agg

,movie_id,tags
0,1,fun | pixar
1,2,fantasy | game | magic board game | robin will...
2,3,moldy | old
3,5,pregnancy | remake
4,7,remake
...,...,...
1567,183611,comedy | funny | rachel mcadams
1568,184471,adventure | alicia vikander | video game adapt...
1569,187593,josh brolin | ryan reynolds | sarcasm
1570,187595,emilia clarke | star wars


In [13]:
tags_agg["tags"] = tags_agg["tags"].apply(lambda x: x.split(" | "))

In [14]:
movies = movies.merge(tags_agg, on="movie_id", how="left")

In [15]:
movies


,movie_id,title,genres,year,avg_rating,rating_count,tags
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,3.920930,215.0,"[fun, pixar]"
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,3.431818,110.0,"[fantasy, game, magic board game, robin williams]"
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,3.259615,52.0,"[moldy, old]"
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,2.357143,7.0,NaN
4,5,Father of the Bride Part II,[Comedy],1995,3.071429,49.0,"[pregnancy, remake]"
...,...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic,"[Action, Animation, Comedy, Fantasy]",2017,4.000000,1.0,NaN
9738,193583,No Game No Life: Zero,"[Animation, Comedy, Fantasy]",2017,3.500000,1.0,NaN
9739,193585,Flint,[Drama],2017,3.500000,1.0,NaN
9740,193587,Bungo Stray Dogs: Dead Apple,"[Action, Animation]",2018,3.500000,1.0,NaN


# TMDb 데이터 붙이기

In [16]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

TMDB_BEARER_TOKEN = os.getenv("TMDB_BEARER_TOKEN")
BASE_URL = "https://api.themoviedb.org/3"

HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_BEARER_TOKEN}",
}

c:\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [17]:
def tmdb_search_movie(query: str, year=None, language="en-US"):
    url = f"{BASE_URL}/search/movie"
    params = {
        "query": query,
        "language": language,
        "include_adult": "false",
    }
    if year:
        params["year"] = year

    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("results", [])
        if not results:
            return None

        # 간단한 우선순위: 정확한 제목 + 연도 근접 + popularity 높은 순
        q = query.lower().strip()

        def score(item):
            title = str(item.get("title", "")).lower().strip()
            original_title = str(item.get("original_title", "")).lower().strip()
            exact = 1 if (title == q or original_title == q) else 0

            release_date = item.get("release_date", "")
            item_year = None
            if isinstance(release_date, str) and len(release_date) >= 4:
                try:
                    item_year = int(release_date[:4])
                except:
                    item_year = None

            year_score = 0
            if year and item_year:
                year_score = -abs(year - item_year)

            popularity = float(item.get("popularity", 0))
            return (exact, year_score, popularity)

        best = sorted(results, key=score, reverse=True)[0]
        return best

    except requests.RequestException as e:
        print(f"[SEARCH ERROR] {query} ({year}): {e}")
        return None

In [18]:
def tmdb_get_movie_details(tmdb_id: int, language="en-US"):
    url = f"{BASE_URL}/movie/{tmdb_id}"
    params = {
        "language": language,
        "append_to_response": "credits,keywords"
    }

    try:
        resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
        resp.raise_for_status()
        return resp.json()
    except requests.RequestException as e:
        print(f"[DETAIL ERROR] {tmdb_id}: {e}")
        return None

In [21]:
import time

def enrich_movies_with_tmdb(df, max_rows=10, sleep_sec=0.25):
    df = df.copy()

    # 결과 컬럼 미리 생성
    add_cols = [
        "tmdb_id", "tmdb_title", "overview", "runtime", "release_date",
        "tmdb_vote_average", "tmdb_vote_count", "director", "cast_top3",
        "tmdb_genres", "poster_path", "popularity"
    ]
    for c in add_cols:
        if c not in df.columns:
            df[c] = None

    # target_idx = df.index[:max_rows] if max_rows else df.index

    processed = 0
    skipped = 0

    for idx, row in df.iterrows():

        if pd.notna(row.get("tmdb_id")):
            skipped += 1
            continue

        # row = df.loc[idx]
        total = len(df)
        title = row.get("title_clean") or row.get("title")
        year = int(row.get("year"))

        if not title:
            continue
        # print ("title", title)
        try:
            search_result = tmdb_search_movie(title, year=year)
            time.sleep(sleep_sec)

            if not search_result:
                print(f"[NO MATCH] {title} ({year})")
                continue

            tmdb_id = search_result.get("id")
            if not tmdb_id:
                print(f"[NO TMDB ID] {title} ({year})")
                continue

            details = tmdb_get_movie_details(tmdb_id)
            time.sleep(sleep_sec)

            if not details:
                continue

            credits = details.get("credits", {})
            cast = credits.get("cast", []) if isinstance(credits, dict) else []
            crew = credits.get("crew", []) if isinstance(credits, dict) else []

            director = None
            for member in crew:
                if member.get("job") == "Director":
                    director = member.get("name")
                    break

            cast_top3 = [c.get("name") for c in cast[:3] if c.get("name")]
            genre_names = [g.get("name") for g in details.get("genres", []) if g.get("name")]

            df.at[idx, "tmdb_id"] = details.get("id")
            df.at[idx, "tmdb_title"] = details.get("title")
            df.at[idx, "overview"] = details.get("overview")
            df.at[idx, "runtime"] = details.get("runtime")
            df.at[idx, "release_date"] = details.get("release_date")
            df.at[idx, "tmdb_vote_average"] = details.get("vote_average")
            df.at[idx, "tmdb_vote_count"] = details.get("vote_count")
            df.at[idx, "director"] = director
            df.at[idx, "cast_top3"] = "|".join(cast_top3) if cast_top3 else None
            df.at[idx, "tmdb_genres"] = "|".join(genre_names) if genre_names else None
            df.at[idx, "poster_path"] = details.get("poster_path")
            df.at[idx, "popularity"] = details.get("popularity")

            print(f"[OK] {row['movie_id']} | {title} ({year}) -> TMDb {tmdb_id}")
            processed += 1
            if processed % 100 == 0:
                df.to_csv("data/movies_enriched.csv", index=False, encoding="utf-8-sig")
                print("💾 자동 저장 완료")
        except Exception as e:
            print(f"[ERROR] {title} ({year}): {e}")
            continue
    print(f"Total: {total}, Processed: {processed}, Skipped (already enriched): {skipped}")


    return df

In [22]:
# movies_enriched = enrich_movies_with_tmdb(movies, max_rows=100, sleep_sec=0.3)
movies_enriched = enrich_movies_with_tmdb(movies, sleep_sec=0.3)
movies_enriched[
    ["movie_id", "title", "year", "tmdb_title", "overview", "runtime", "director"]
].head(10)

[OK] 1 | Toy Story (1995) -> TMDb 862
[OK] 2 | Jumanji (1995) -> TMDb 8844
[OK] 3 | Grumpier Old Men (1995) -> TMDb 15602
[OK] 4 | Waiting to Exhale (1995) -> TMDb 31357
[OK] 5 | Father of the Bride Part II (1995) -> TMDb 11862
[OK] 6 | Heat (1995) -> TMDb 949
[OK] 7 | Sabrina (1995) -> TMDb 11860
[OK] 8 | Tom and Huck (1995) -> TMDb 45325
[OK] 9 | Sudden Death (1995) -> TMDb 9091
[OK] 10 | GoldenEye (1995) -> TMDb 710
[OK] 11 | American President, The (1995) -> TMDb 9087
[OK] 12 | Dracula: Dead and Loving It (1995) -> TMDb 12110
[OK] 13 | Balto (1995) -> TMDb 21032
[OK] 14 | Nixon (1995) -> TMDb 10858
[OK] 15 | Cutthroat Island (1995) -> TMDb 1408
[OK] 16 | Casino (1995) -> TMDb 524
[OK] 17 | Sense and Sensibility (1995) -> TMDb 4584
[OK] 18 | Four Rooms (1995) -> TMDb 5
[OK] 19 | Ace Ventura: When Nature Calls (1995) -> TMDb 9273
[OK] 20 | Money Train (1995) -> TMDb 11517
[OK] 21 | Get Shorty (1995) -> TMDb 8012
[OK] 22 | Copycat (1995) -> TMDb 1710
[OK] 23 | Assassins (1995) -> TMDb

KeyboardInterrupt: 

LangChain Document 생성

In [26]:
movies_enriched.count

<bound method DataFrame.count of       movie_id                               title  \
0            1                           Toy Story   
1            2                             Jumanji   
2            3                    Grumpier Old Men   
3            4                   Waiting to Exhale   
4            5         Father of the Bride Part II   
...        ...                                 ...   
9737    193581  Black Butler: Book of the Atlantic   
9738    193583               No Game No Life: Zero   
9739    193585                               Flint   
9740    193587        Bungo Stray Dogs: Dead Apple   
9741    193609        Andrew Dice Clay: Dice Rules   

                                                 genres  year  avg_rating  \
0     [Adventure, Animation, Children, Comedy, Fantasy]  1995    3.920930   
1                        [Adventure, Children, Fantasy]  1995    3.431818   
2                                     [Comedy, Romance]  1995    3.259615   
3         

In [24]:
from langchain_core.documents import Document 

docs = []

for _, row in movies_enriched.iterrows():
    page_content = f"""
Movie title: {row.get('tmdb_title') or row.get('title')}
Release year: {row.get('year')}
Genres: {row.get('tmdb_genres') or ', '.join(row.get('genres_list', []))}
Overview: {row.get('overview') or ''}
Director: {row.get('director') or ''}
User tags: {row.get('tags', '')}
Cast: {row.get('cast_top3') or ''}
Average rating: {row.get('avg_rating', '')}
Rating count: {row.get('rating_count', '')}
Runtime: {row.get('runtime', '')} minutes
"""

    metadata = {
        "movie_id": row.get("movie_id"),
        "title": row.get("tmdb_title") or row.get("title"),
        "year": row.get("year"),
        "genres": row.get("genres_list"),
        "avg_rating": row.get("avg_rating"),
        "rating_count": row.get("rating_count"),
        "runtime": row.get("runtime"),
        "director": row.get("director"),
    }

    docs.append(Document(page_content=page_content, metadata=metadata))

In [35]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

C:\Users\김소영\AppData\Local\Temp\ipykernel_1004\3340972191.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\김소영\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("faiss_movies")

c:\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


NameError: name 'docs' is not defined

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
loaded_db = FAISS.load_local(
    "faiss_movies",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

C:\Users\김소영\AppData\Local\Temp\ipykernel_14984\672310219.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [10]:
print(loaded_db.index.ntotal)

9742


In [12]:
results = loaded_db.similarity_search(
    "sifi movie with space battles and futuristic technology",
    k=5
)

for r in results:
    print(r.page_content)


Movie title: Space Battleship Yamato
Release year: 2010
Genres: 
Overview: 
Director: 
User tags: nan
Cast: 
Average rating: 3.0
Rating count: 1.0
Runtime: None minutes


Movie title: Last Starfighter, The
Release year: 1984
Genres: 
Overview: 
Director: 
User tags: nan
Cast: 
Average rating: 3.5714285714285716
Rating count: 7.0
Runtime: None minutes


Movie title: Armageddon
Release year: 1998
Genres: 
Overview: 
Director: 
User tags: ['space']
Cast: 
Average rating: 3.0543478260869565
Rating count: 92.0
Runtime: None minutes


Movie title: Stargate
Release year: 1994
Genres: 
Overview: 
Director: 
User tags: ['time travel']
Cast: 
Average rating: 3.375
Rating count: 140.0
Runtime: None minutes


Movie title: Star Trek
Release year: 2009
Genres: 
Overview: 
Director: 
User tags: ['future', 'lack of development', 'lack of story', 'quick cuts', 'sci-fi', 'simon pegg', 'space', 'space travel', 'time travel']
Cast: 
Average rating: 3.864406779661017
Rating count: 59.0
Runtime: None minut